In [3]:
l = [-1, -1, -1, -1, 0]

print(str(l))

endpoint_listprint(''.join([str(i) for i in l]))

[-1, -1, -1, -1, 0]
-1-1-1-10


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim

class MyModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MyModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [20]:
import time
from typing import Dict, Optional, Any

import ray
from ray import train, tune
from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.optuna import OptunaSearch

from ray.train.torch import TorchTrainer, get_device

ray.shutdown()
ray.init(configure_logging=False)

def evaluate(step, width, height, activation):
    time.sleep(0.1)
    device  = get_device()
    model = MyModel(10, int(width), 2)
    model.to(device)
    activation_boost = 10 if activation=="relu" else 0
    return (0.1 + width * step / 100) ** (-1) + height * 0.1 + activation_boost


def objective(config):
    for step in range(config["steps"]):
        score = evaluate(step, config["width"], config["height"], config["activation"])
        train.report({"iterations": step, "mean_loss": score})


search_space = {
    "steps": 100,
    "width": tune.uniform(0, 20),
    "height": tune.uniform(-100, 100),
    "activation": tune.choice(["relu", "tanh"]),
}

#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
algo = OptunaSearch()
algo = ConcurrencyLimiter(algo, max_concurrent=4)
num_samples = 10


tuner = tune.Tuner(
    objective,
    tune_config=tune.TuneConfig(
        metric="mean_loss",
        mode="min",
        search_alg=algo,
        num_samples=num_samples,
    ),
    param_space=search_space,
)
results = tuner.fit()

device(type='cuda', index=0)

In [10]:
df = results.get_dataframe()
df

,iterations,mean_loss,timestamp,checkpoint_dir_name,done,training_iteration,trial_id,date,time_this_iter_s,time_total_s,pid,hostname,node_ip,time_since_restore,iterations_since_restore,config/steps,config/width,config/height,config/activation,logdir
0,99,4.545164,1707165485,None,False,100,eb89f77c,2024-02-05_21-38-05,0.100745,10.136045,70516,XPS-15-9570,127.0.0.1,10.136045,100,100,16.920281,44.858206,tanh,eb89f77c
1,99,-9.813428,1707165490,None,False,100,2ca83541,2024-02-05_21-38-10,0.102700,10.148129,68132,XPS-15-9570,127.0.0.1,10.148129,100,100,8.061771,-99.371728,tanh,2ca83541
2,99,1.567462,1707165495,None,False,100,161d339a,2024-02-05_21-38-15,0.102313,10.175425,51508,XPS-15-9570,127.0.0.1,10.175425,100,100,6.257070,-85.914066,relu,161d339a
3,99,1.546969,1707165496,None,False,100,8aabcdac,2024-02-05_21-38-16,0.100625,10.158978,70516,XPS-15-9570,127.0.0.1,10.158978,100,100,16.636113,14.866184,tanh,8aabcdac
4,99,-0.130324,1707165500,None,False,100,c80376a7,2024-02-05_21-38-20,0.101606,10.172995,79248,XPS-15-9570,127.0.0.1,10.172995,100,100,14.034814,-2.017804,tanh,c80376a7
5,99,0.487969,1707165501,None,False,100,910fcac1,2024-02-05_21-38-21,0.100642,10.171946,68132,XPS-15-9570,127.0.0.1,10.171946,100,100,6.387008,-96.677186,relu,910fcac1
6,99,16.441872,1707165506,None,False,100,a6fddbc3,2024-02-05_21-38-26,0.102404,10.163965,51508,XPS-15-9570,127.0.0.1,10.163965,100,100,13.619315,63.682507,relu,a6fddbc3
7,99,19.910979,1707165507,None,False,100,7f25b6b0,2024-02-05_21-38-27,0.101806,10.163839,70516,XPS-15-9570,127.0.0.1,10.163839,100,100,8.677803,97.959179,relu,7f25b6b0
8,99,1.323914,1707165511,None,False,100,37771520,2024-02-05_21-38-31,0.100938,10.181928,79248,XPS-15-9570,127.0.0.1,10.181928,100,100,9.611013,12.199086,tanh,37771520
9,99,9.466051,1707165511,None,False,100,041f6e63,2024-02-05_21-38-31,0.103150,10.190334,68132,XPS-15-9570,127.0.0.1,10.190334,100,100,19.607306,94.147986,tanh,041f6e63
